In [ ]:
from typing import Tuple

import joblib
import mlflow.sklearn
import pandas as pd
from core.mlops.mlflow_aux import set_local_mlflow_tracking_uri
from mlflow.tracking import MlflowClient

tracking_uri = set_local_mlflow_tracking_uri(project_name="classificador-cbo")
client = MlflowClient()


def load_preprocesser_data_and_model(
    model_name: str, prod_name: str
) -> Tuple[pd.DataFrame]:
    model_uri = f"models:/{model_name}@{prod_name}"

    # Pega os detalhes da versão do modelo
    model_details = client.get_model_version_by_alias(name=model_name, alias=prod_name)
    run = mlflow.get_run(run_id=model_details.run_id)

    dataframe_path = client.download_artifacts(run_id=run.info.run_id, path=".joblib")
    with open(dataframe_path, "rb") as joblib_f:
        df = joblib.load(joblib_f)

    dataframe_path = client.download_artifacts(
        run_id=run.info.run_id, path="fitting_data.joblib"
    )
    with open(dataframe_path, "rb") as joblib_f:
        df = joblib.load(joblib_f)

    # Carrega o modelo em memória
    model = mlflow.sklearn.load_model(model_uri)

    return df, model


model_name = "tfidf_vect"
prod_name = "champion"